# Tutorial: An Introduction to Complete Active Space Self-Consistent Field (CASSCF) Theory

---

This tutorial motivates and introduces the complete active space self-consistent field (CASSCF) method.
You will explore the challenges of describing the electronic structure of molecules in bond-breaking processes.
This tutorial assumes that you are already familiar with Hartree–Fock theory or Density Functional Theory (DFT) and that you have some experience running quantum chemistry calculations.

This tutorial includes assigned tasks (📝) that you should complete as you work through the material.
These tasks are designed to help you understand the concepts and techniques presented in the tutorial and to give you hands-on experience with running CASSCF calculations using Forte2.

## 📚 The challenge of describing bond-breaking processes

In this tutorial we will use the N<sub>2</sub> molecule as an illustrative example of the challenges of describing bond-breaking processes with quantum chemistry methods.
The N<sub>2</sub> molecule has a triple bond with an experimental bond length of 1.09768 Å and a dissociation energy (D<sub>0</sub>) of ca. 9.76 eV (224.99 kcal/mol).

---

### 📝 **Tasks**
1. Using your code of choice, write a Python script or Jupyter notebook to compute and plot the potential energy curve of N<sub>2</sub> as a function of the bond length using Hartree–Fock or DFT. Use a cc-pVDZ basis and sample the bond length from 0.8 Å to 3.0 Å in increments of 0.1 Å.
2. Calculate the equilibrium dissociation energy (D<sub>e</sub>) of N<sub>2</sub> by computing difference between the energy at the equilibrium geometry and the dissociated limit (e.g., at 3.0 Å).
3. What is the relationship between D<sub>0</sub> and D<sub>e</sub>? How big do you expect the difference to be compared to the bond dissociation energy?
4. Using experimental data, convert the experimental D<sub>0</sub> to D<sub>e</sub> and compare with D<sub>e</sub> from your computation. Discuss any discrepancies between the computed and experimental D<sub>e</sub> values.

---

## 🧸 A toy problem: the hydrogen molecule

Before we start examining the N<sub>2</sub> molecule, we will consider a simpler system: the H<sub>2</sub> molecule described by a minimal basis set. Using this toy problem, we will be able to motivate why the conventional molecular orbital description of the electronic structure of molecules breaks down in bond-breaking processes and what ingredients are needed to describe these processes accurately.

We consider the case of a minimal basis set that contains 1s atomic orbital for each hydrogen atom. Therefore, the H<sub>2</sub> molecule has two electrons and two molecular orbitals. These orbitals can be classified as:
- A bonding orbital ($\sigma_{1s}$), given by the plus combination of the two 1s atomic orbitals.
- An antibonding orbital ($\sigma^*_{1s}$), given by the minus combination of the two 1s atomic orbitals.

The following figure shows how the two molecular orbitals of H<sub>2</sub> are constructed from the two 1s atomic orbitals. The bonding orbital ($\sigma_{1s}$) is given by the plus combination of the two atomic orbitals, while the antibonding orbital ($\sigma^*_{1s}$) is given by the minus combination of the two atomic orbitals.

<center><img src="img/h2_mo_diagram.png" alt="Molecular orbital diagram of H2" width="400"/></center>

### 💻 Setting up a computation in Forte2
Let's compute these with Forte2 and plot the potential energy curve of H<sub>2</sub> as a function of the bond length. We will use a small basis set (STO-6G) for this calculation.

To begin we will import the NumPy library and import a few components from Forte2:

In [ ]:
import numpy as np
from forte2 import System, RHF, State, CI, CISolver, MCOptimizer, write_orbital_cubes

Setting up a computation in Forte2 requires defining a `System` object that contains the molecular geometry and basis set information, and then using this system to perform a Hartree-Fock computation.
We then create a restricted Hartree-Fock (`RHF`) object, specify the charge of the system (0), and connect this to the `System` object we created.
The resulting object is stored in the variable `rhf`. 
In the last step, we call the `run()` method to perform the computation on the `rhf` object.

In [ ]:
# equilibrium geometry
xyz = """H 0.0 0.0 0.0
H 0.0 0.0 0.74"""

# create a System object. cholesky_tei = True approximates the two-electron integrals with the Cholesky decomposition
system = System(xyz=xyz, basis_set="sto-6g", cholesky_tei=True)

# setup a restricted Hartree-Fock (RHF) calculation and run it
scf = RHF(charge=0)(system)
scf.run()

The last lines in the output show that the composition of the molecular orbitals. For example, the first molecular orbital (MO 0) is given by the combination of the two 1s atomic orbitals on hydrogen atom 1 and 2 with a coefficient of -0.5489:
```
# MO  (AO) label : coeff                      
0     H1 1s (0): -0.5489        H2 1s (1): -0.5489       
```
In other words, the bonding orbital ($\sigma_{1s}$) is given by the following linear combination of atomic orbitals:
\begin{equation}
\sigma_{1s} = -0.5489 \, (\psi_\text{1s}^{\text{H}_1} + \psi_\text{1s}^{\text{H}_2}).
\end{equation}
Note that the overall sign of the orbital is arbitrary and that on your computer it might be of the opposite sign (i.e., +0.5489 instead of -0.5489).

---

**Study question:** Why is the sign of the orbital irrelevant? Consider an electron in 1 dimension ($x$). Show that given an operator like the Hamiltonian, the expectation value of the orbital $\psi(x)$ is the same if you multiply the sign of an orbital by a factor of -1.

### 💻 The potential energy curve of H<sub>2</sub>

To compute the potential energy curve of H<sub>2</sub>, we will create a function that takes the bond length as input and returns the energy of the system at that bond length. We will then use this function to compute the energy at different bond lengths and plot the results.

In [ ]:
def compute_h2_energy(r):
    xyz = f"""H 0.0 0.0 0.0
            H 0.0 0.0 {r}"""
    system = System(xyz=xyz, basis_set='STO-6G',auxiliary_basis_set='cc-pVTZ-JKFit')
    rhf = RHF(charge=0)(system)
    rhf.run()
    return rhf.E

# equilibrium bond length of H2 in Angstroms
re_h2 = 0.74144

Let's verify that this function returns the correct energy at a bond length of 1.0 Å:

In [ ]:
ref_energy = -1.073584 # reference value at 1.0 Å

energy_at_1A = compute_h2_energy(1.0)
print(f"Energy at 1.0 Å:  {energy_at_1A:.6f} Hartree")
print(f"Reference energy: {ref_energy:.6f} Hartree")    

Next, to plot the potential energy curve, we set up a range of bond lengths from 0.6 to 3.0 Å and compute the energy at each bond length using the `compute_energy` function.

In [ ]:
# define a range of bond lengths from 0.5 to 3.0 Å and 0.1 Å increments
r_vals = np.linspace(0.5, 3.0, 26)

# compute the energy at each bond length and store the results in a list
energy_vals = [compute_h2_energy(r) for r in r_vals]

In [ ]:
# plot the results using matplotlib
import matplotlib.pyplot as plt
plt.plot(r_vals, energy_vals, marker='o')
plt.xlabel('H-H distance (Å)')
plt.ylabel('Energy (Hartree)')
plt.title('Hartree-Fock Energy of H$_2$ vs. bond length')
plt.show()

This curve looks like we would expect. At the equilibrium bond length of 0.74 Å, the energy is at a minimum. As we stretch the bond, the energy increases. However, if we compute the equilibrium dissociation energy (D<sub>e</sub>) we get the following value:

In [ ]:
De_hartree = compute_h2_energy(5.0) - compute_h2_energy(re_h2)
print(f"\nEquilibrium dissociation energy (D_e) of H2: {De_hartree * 627.509:.6f} kcal/mol")

The computed D<sub>e</sub> (324.6 kcal/mol) is much larger than the experimental D<sub>0</sub> (102.9 kcal/mol).

This discrepancy arises because Hartree-Fock does not account for **electron correlation**, which is particularly important in bond-breaking processes. As we stretch the bond, the electrons become more correlated and the Hartree-Fock method fails to capture this effect, overestimating the dissociation energy.

## 📚 Going beyond Hartree-Fock theory: multiconfigurational methods

To address the limitations of Hartree–Fock theory we need to move to a multiconfigurational description of the electronic structure. In Hartree–fock theory, we approximate the wavefunction as a single Slater determinant, which corresponds to a single configuration of electrons in the molecular orbitals. Specifically, we can write the Hartree–Fock wavefunction for H<sub>2</sub> ($\Psi_\text{HF}$) as:
\begin{equation}
| \Psi_\text{HF} \rangle = |(\sigma_{1s})^2 \rangle,
\end{equation}
where we use the notation $|(\sigma_{1s})^2 \rangle$ to indicate that both electrons are occupying the bonding orbital ($\sigma_{1s}$).
The exact wavefunction for H<sub>2</sub> can be written as a linear combination of all possible configurations of electrons in the molecular orbitals with corresponding coefficients.
For the minimal basis set, we have three possible configurations:

<center><img src="img/h2_fci.png" alt="Configurations of H2" width="400"/></center>

The second configuration ($|(\sigma_{1s})^1(\sigma^*_{1s})^1 \rangle$) corresponds to a single substitution of the Hartree–Fock determinant, and has two determinants contributing to it, which are combined together to form a singlet state. The third configuration ($|(\sigma^*_{1s})^2 \rangle$) corresponds to a double substitution of the Hartree–Fock determinant, and has only one determinant contributing to it.

The exact wavefunction can be written as a linear combination of these three configurations:
\begin{equation}
| \Psi_\text{exact} \rangle = c_1 |(\sigma_{1s})^2 \rangle + c_2 |(\sigma_{1s})^1(\sigma^*_{1s})^1 \rangle + c_3 |(\sigma^*_{1s})^2 \rangle,
\end{equation}
and by varying the coefficients $c_1$, $c_2$, and $c_3$ we can obtain the exact wavefunction that describes H<sub>2</sub> at all bond lengths, including the dissociation limit.
This procedure is called a **full configuration interaction (FCI)** calculation.

### 💻 Setting up a FCI computation for H<sub>2</sub>
To perform an FCI calculation on H<sub>2</sub>, we can use the `CI` class in Forte2. This class will take the Hartree–Fock object as input and perform a full configuration interaction calculation to obtain the exact wavefunction and energy of the system.

To set up the FCI computation, we have to first define the target electronic state.
In our case we select the singlet state (`multiplicity = 1`) and
\begin{equation}
M_S = \frac{1}{2} (N_\alpha - N_\beta),
\end{equation}
where $N_\alpha$ and $N_\beta$ are the number of alpha and beta electrons, respectively. For a singlet state, we have an equal number of alpha and beta electrons, so $M_S = 0$, which we set via `ms=0.0`.

Finally, we have to indicate how many orbitals will be included in the FCI treatment (active space). In this case, we have two molecular orbitals (the bonding and antibonding orbitals), so we set `active_orbitals=2`.

In [ ]:
# equilibrium geometry
xyz = """H 0.0 0.0 0.0
H 0.0 0.0 0.74"""

system = System(xyz=xyz, basis_set="sto-6g", cholesky_tei=True)
scf = RHF(charge=0)(system)

# specify the target FCI electronic state with multiplicity 1 (singlet) and ms=0.0
singlet = State(system=system,multiplicity=1,ms=0.0)

# create a CI object specifying the state number of active orbitals (2)
fci = CI(states=[singlet],
         active_orbitals=2)(scf)
fci.run()

The FCI calculation will return the total energy (-1.1459398103 Hartree) and a summary showing the top determiants:
```
==============================================================
Contrib.  #1           #2           #3           #4           
--------------------------------------------------------------
Root 0    |20>         |02>         |ab>         |ba>         
          -0.993640    +0.112601    -0.000000    -0.000000    
==============================================================
```
At this geometry, the FCI wavefunction is dominated by the Hartree–Fock configuration ($|(\sigma_{1s})^2 \rangle$ = `|20>`) with a coefficient of -0.993640, and has a smaller contribution (+0.112601) from the double substitution configuration ($|(\sigma^*_{1s})^2 \rangle$ = `|02>`).

Notice that the two determinants that contribute to the single substitution configuration ($|(\sigma_{1s})^1(\sigma^*_{1s})^1 \rangle$ = `|ab>` and `|ba>`) have zero contribution to the wavefunction at this geometry. This is because this configuration has a different spatial symmetry than the ground state of H<sub>2</sub> and therefore cannot mix with the other configurations.


---

### 📝 **Tasks**.
1. Start from the code in the previous cell and define a Python function that computes the FCI energy of H<sub>2</sub> at a given bond length using FCI.
2. Use this function to compute the FCI potential energy curve of H<sub>2</sub> and make a plot comparing the HF and FCI results.
3. Compute the equilibrium dissociation energy and compare it with the experimental value. Discuss any discrepancies between the computed and experimental D<sub>e</sub> values.
4. Analyze the composition of the FCI wavefunction at bond lengths of 0.75, 1.5, and 3.0 Å. How does the contribution of the different configurations change as we stretch the bond? What does this tell us about the nature of the electronic structure of H<sub>2</sub> at different bond lengths?

---

## 📚 The complete active space self-consistent field approach

While FCI is the exact solution to the electronic Schrödinger equation within a given basis set, it quickly becomes too expensive as the number of electrons and orbitals increases.

An alternative is to combined FCI in a small set of orbitals (called the active space) with an optimization of the molecular orbitals to minimize the energy. This is the idea behind the complete active space self-consistent field (CASSCF) approach.

The following figure shows the terminology used in CASSCF to describe the different sets of orbitals:

<center><img src="img/active_space.png" alt="Active Space Terminology" width="300"/></center>

We distinguish between three types of orbitals:
- **Core orbitals**: these orbitals that are always doubly occupied in all CASSCF determinants. Usually chosen to be the lowest-energy orbitals.
- **Active orbitals**: these orbitals can have different occupations in the CASSCF determinants. These orbital are involved in the chemical process of interest (e.g., $\sigma_\mathrm{1s}$ and $\sigma^*_\mathrm{1s}$ orbitals for H<sub>2</sub>).
- **Virtual orbitals**: these orbitals are always unoccupied in all CASSCF determinants.

In CASSCF the wavefunction is written as:
\begin{equation}
| \Psi_\text{CASSCF} \rangle = \sum_I c_I | \Phi_I \rangle,
\end{equation}
where the Slater determinants $| \Phi_I \rangle$ are generated by distributing a given number of electrons in the active orbitals. Both the orbitals that enter in the CASSCF determinants and the coefficients $c_I$ are optimized to minimize the energy of the system.

To run a CASSCF calculation in Forte2, we can use the `CISolver` class to define the active space and the `MCOptimizer` class to optimize the orbitals and coefficients. The `CISolver` class will generate the configurations for the specified active space, while the `MCOptimizer` class will perform the optimization of the orbitals and coefficients to minimize the energy.

The following example shows how to set up a CASSCF calculation for H<sub>2</sub> using an active space containing two orbitals, and a double-zeta basis set (cc-pVDZ):

In [ ]:
# equilibrium geometry
xyz = """H 0.0 0.0 0.0
H 0.0 0.0 0.74"""

system = System(xyz=xyz, basis_set="cc-pVDZ", cholesky_tei=True)
scf = RHF(charge=0)(system)
singlet = State(system=system,multiplicity=1,ms=0.0)

# create a CISolver object specifying the state number of active orbitals (2)
casci = CISolver(states=[singlet],active_orbitals=2)
# create a CASSCF object specifying the state number of active orbitals (2) and the maximum number of iterations (1000)
casscf = MCOptimizer(casci)(scf)

casscf.run()

📝 **Tasks**. After running the CASSCF computation on H<sub>2</sub>, complete the following tasks:
1. Compute the CASSCF potential energy curve of H<sub>2</sub> on a grid in the range 0.5 to 3.0 Å with 0.1 Å spacing.
2. Fit a Morse potential to the CASSCF energy points and plot the resulting curve together with the CASSCF points.
3. From the Morse potential fit, extract the equilibrium bond length and dissociation energy. Compare these values with the experimental values and discuss any discrepancies.

## 💻 The nitrogen molecule revisited: setting up a CASSCF computation

### Choosing the active space

In analogy to the case of H<sub>2</sub>, you can probably guess that CASSCF can be used to correctly describe the dissociation of N<sub>2</sub>. However, before we set up a calculation, we need to answer the question: **which orbitals should be included in the active space?**

To answer this question, we need to identify which orbitals are involved in the bond-breaking process.
The following diagram shows the valence atomic orbitals (AOs) of nitrogen (2s and 2p) and the resulting molecular orbitals (MOs) of N<sub>2</sub>:

<center><img src="img/n2_mo_diagram.png" alt="MO diagram for N2" width="600"/></center>

In this case the orbitals involved in the bond-breaking process are the bonding and antibonding orbitals that arise from the 2p atomic orbitals.
While the 2s orbitals are also involved in the electronic structure of N<sub>2</sub>, their net contribution is nonbonding (one is bonding, one is antibonding). Therefore, as a first approximation, we can exclude them from the active space for describing the bond-breaking process (they can be later reintroduced to determine how they impact the results).

Note that here we have used the more conventional notation for the MOs, specifying the type ($\sigma$ or $\pi$) and the parity with respect to inversion (g or u). Also, notice that the the $\pi$ orbitals are doubly degenerate, so we have two $1\pi_\mathrm{u}$ orbitals and two $1\pi^*_\mathrm{g}$ orbitals.

The following code sets up a CASSCF calculation for N<sub>2</sub> using an active space containing the six orbitals arising from the 2p atomic orbitals (the $3\sigma_\mathrm{g}$, $1\pi_\mathrm{u}$, $1\pi^*_\mathrm{g}$, and $3\sigma_\mathrm{u}$ orbitals), and a double-zeta basis set (cc-pVDZ).
In this computation, the four orbitals that arise from the 1s and 2s atomic orbitals are treated as core orbitals.

In [ ]:
# equilibrium bond length of N2 in Angstroms
re_n2 = 1.097685 

# equilibrium geometry
xyz = f"""N 0.0 0.0 0.0
N 0.0 0.0 {re_n2}"""

system = System(xyz=xyz, basis_set="cc-pVDZ", cholesky_tei=True)
scf = RHF(charge=0)(system)
singlet = State(system=system,multiplicity=1,ms=0.0)
# Set up a CAS(6e,6o) computation
casci = CISolver(states=[singlet],
                 core_orbitals=4,
                 active_orbitals=6)
casscf = MCOptimizer(casci)(scf)
casscf.run()

# Check that the computed energy matches the expected value
assert abs(casscf.E - (-109.0900240696)) < 1e-6, f"Expected energy of -109.090024 Hartree, but got {casscf.E:.6f} Hartree"

If all goes well, the CASSCF computation should converge in a few iterations and give you an energy of -109.090024 Hartree.

### 🌌 Visualizing the CASSCF orbitals

Before we declare victory, we need to verify that the optimized CASSCF orbitals are indeed the ones we intended to include. To do this, we can visualize the CASSCF orbitals and check that they correspond to the expected MOs.

You can generate molecular orbitals in the cube file format using the function `write_orbital_cubes`.
This function takes as inputs:
- a System object (the `system` we defined above)
- the orbital coefficient matrix (e.g., `casscf.C[0]`)
- a list of orbital indices (here set to the first 10 orbitals)
- an optional `filepath` argument to indicate where to save the cube files (default is current working directory)

⚠️ **Forte2 follows these conventions for ordering the orbitals**:
- the orbitals follow the order: core, active, virtual
- the orbitals are numbered starting from 0. So the first orbital has index 0, the second has index 1, and so on

Therefore, in our case, the first 4 orbitals (indices 0 to 3) correspond to the core orbitals, while the next 6 orbitals (indices 4 to 9) correspond to the active orbitals.

---
### 📝 **Tasks**
- Run the code in the cell below to generate cubes for the first 10 CASSCF orbitals.
- Visualize the generated cube files and verify that they correspond to the expected MOs.
---

In [ ]:
!mkdir -p casscf_orbitals
write_orbital_cubes(system=system, C=casscf.C[0],indices=range(10),filepath='casscf_orbitals')

## 💻 Computing the potential energy curve of N<sub>2</sub> with CASSCF

To end this tutorial, we will compute the potential energy curve of N<sub>2</sub> as a function of the bond length using CASSCF. We will use the same active space and basis set as in the previous section.

The next two cells define a function `compute_n2_energy` that takes the bond length as input and returns the CASSCF energy of N<sub>2</sub> at that bond length.
The second cell uses this function to compute the energy at different bond lengths and plot the results.

In [ ]:
def compute_n2_energy(r):
    """
    Compute the energy of N_2 at a given bond length r (in Angstroms) using CASSCF.
    """
    
    xyz = f"""N 0.0 0.0 0.0;N 0.0 0.0 {r}"""
    system = System(xyz=xyz, basis_set="cc-pVDZ", cholesky_tei=True)
    scf = RHF(charge=0)(system)
    casci = CISolver(states=[State(system=system,multiplicity=1,ms=0.0)],
                     core_orbitals=4,
                     active_orbitals=6)
    casscf = MCOptimizer(casci)(scf)
    casscf.run()
    return casscf.E

In [ ]:
r_vals = np.linspace(0.85, 2.5, 34)
energy_vals = [compute_n2_energy(r) for r in r_vals]

plt.plot(r_vals, energy_vals, marker='o')
plt.xlabel('N-N distance (Å)')
plt.ylabel('Energy (Hartree)')
plt.title('CASSCF Energy of N$_2$ vs. bond length')
plt.show()

## 🧭 End of the tutorial and next steps

This tutorial introduced you to the complete active space self-consistent-field (CASSCF) method.
However, it has only scratched the surface of what CASSCF can do and how it works. There are many more topics that we have not covered in this tutorial, such as:
- How to choose the active space for a given problem.
- How to perform state-averaged CASSCF calculations to describe excited states.
- How to use CASSCF as a reference for multireference perturbation theory (e.g., DSRG-MRPT2) to include dynamic correlation effects.

These can all be explored using Forte2, starting from representative computations found in the test cases. You can also consult the references below to learn more about CASSCF and its applications.

## References

Here are some references that you can consult to learn more about CASSCF and its applications:

1. Schmidt, M. W.; Gordon, M. S. The Construction and Interpretation of MCSCF Wavefunctions. *Annu. Rev. Phys. Chem.* **1998**, 49, 233–266. https://doi.org/10.1146/annurev.physchem.49.1.233.
2. Veryazov, V.; Malmqvist, P. Å.; Roos, B. O. How to Select Active Space for Multiconfigurational Quantum Chemistry? *Int. J. of Quantum Chemistry* **2011**, 111, 3329–3338. https://doi.org/10.1002/qua.23068.
3. C. D. Sherrill; H. F. Schaefer III. The Configuration Interaction Method: Advances in Highly Correlated Approaches. *Advances in Quantum Chemistry*; Academic Press, 1999; Vol. 34, pp 143–269. https://doi.org/10.1016/S0065-3276(08)60532-8.
4. Szalay, P. G.; Müller, T.; Gidofalvi, G.; Lischka, H.; Shepard, R. Multiconfiguration Self-Consistent Field and Multireference Configuration Interaction Methods and Applications. *Chem. Rev.* **2012**, 112, 108–181. https://doi.org/10.1021/cr200137a.

